# Tiki Reviews – Data Cleaning Pipeline
Mỗi cell = một bước. Chạy **từ trên xuống** hoặc chạy từng cell độc lập.

| Bước | Nội dung |
|------|----------|
| 0 | Import & Config |
| 1 | Load file CSV |
| 2 | Xóa duplicate |
| 3 | Chuẩn hóa kiểu dữ liệu |
| 4 | Làm sạch text |
| 5 | Xử lý missing values |
| 6 | Lọc outlier / bất thường |
| 7 | Feature engineering |
| 8 | Xuất file sạch |

## Bước 0 · Import & Config

In [59]:
import pandas as pd
import numpy as np
import re
import os
from datetime import datetime
from IPython.display import display

# ── CONFIG ──────────────────────────────────────────
INPUT_FILE      = 'tiki_reviews_dataset.csv'
OUTPUT_CLEAN    = 'tiki_reviews_clean.csv'
OUTPUT_REPORT   = 'tiki_clean_report.txt'

MIN_CONTENT_LEN = 5
MAX_CONTENT_LEN = 5000
VALID_RATINGS   = {1, 2, 3, 4, 5}

print(' Import & config xong')
print(f'   Input : {INPUT_FILE}')
print(f'   Output: {OUTPUT_CLEAN}')


 Import & config xong
   Input : tiki_reviews_dataset.csv
   Output: tiki_reviews_clean.csv


## Bước 1 · Load file CSV

In [60]:
df = pd.read_csv(INPUT_FILE, dtype=str, encoding='utf-8-sig')

print(f'Dòng   : {len(df):,}')
print(f'Cột    : {df.shape[1]}  →  {list(df.columns)}')

# ── Missing ban đầu
miss = df.isna().sum()
miss = miss[miss > 0].rename('missing_count').to_frame()
miss['%'] = (miss['missing_count'] / len(df) * 100).round(1)
print('\nMissing value ban đầu:')
display(miss)

# ── Preview
print('\nMẫu 5 dòng đầu:')
display(df.head())


Dòng   : 15,351
Cột    : 19  →  ['product_id', 'product_name', 'brand_id', 'brand_name', 'price', 'list_price', 'discount', 'discount_rate', 'review_count', 'order_count', 'review_id', 'rating', 'title', 'content', 'thank_count', 'created_at', 'customer_id', 'customer_name', 'purchased_at']

Missing value ban đầu:


,missing_count,%
order_count,15351,100.0
content,9717,63.3
customer_name,10,0.1
purchased_at,16,0.1



Mẫu 5 dòng đầu:


,product_id,product_name,brand_id,brand_name,price,list_price,discount,discount_rate,review_count,order_count,review_id,rating,title,content,thank_count,created_at,customer_id,customer_name,purchased_at
0,279159356,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,18802,Samsung,690000,999000,309000,31,5,NaN,20237513,5,Cực kì hài lòng,Phù hợp với nhu cầu theo dõi sức khỏe và vận đ...,0,1780306629,28031939,Nguyen Quoc Phong,1780016826
1,279159356,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,18802,Samsung,690000,999000,309000,31,5,NaN,20215877,5,Cực kì hài lòng,"Đóng gói cẩn thận, hàng tốt, dễ kết nối",0,1772260479,6450350,PT PHƯƠNG,1772254541
2,279159356,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,18802,Samsung,690000,999000,309000,31,5,NaN,20230022,5,Cực kì hài lòng,Tốt trong tầm giá.,0,1777447322,5263165,Lê Tuấn,1774950204
3,279159356,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,18802,Samsung,690000,999000,309000,31,5,NaN,20234702,5,Cực kì hài lòng,NaN,0,1779145123,721906,Nguyễn Minh Quân,1778844168
4,279159356,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,18802,Samsung,690000,999000,309000,31,5,NaN,20216935,5,Cực kì hài lòng,Good good,0,1772618716,30484861,Hoàng Duy,1772379615


## Bước 2 · Xóa Duplicate

In [61]:
print(f'Trước: {len(df):,} dòng')

# 2a. Trùng hoàn toàn
b = len(df)
df = df.drop_duplicates()
print(f'  ✂ Trùng hoàn toàn     : -{b - len(df):,} dòng')

# 2b. Trùng review_id
dup = df[df.duplicated('review_id', keep=False)]
if not dup.empty:
    print(f'\n  Ví dụ review_id trùng ({len(dup):,} dòng):')
    display(dup[['review_id','product_id','rating','content']].head(6))

b = len(df)
df = df.drop_duplicates(subset=['review_id'], keep='first')
print(f'  ✂ Trùng review_id     : -{b - len(df):,} dòng  (giữ lần đầu)')

df = df.reset_index(drop=True)
print(f'\nSau : {len(df):,} dòng')


Trước: 15,351 dòng
  ✂ Trùng hoàn toàn     : -1 dòng
  ✂ Trùng review_id     : -0 dòng  (giữ lần đầu)

Sau : 15,350 dòng


## Bước 3 · Chuẩn hóa kiểu dữ liệu

In [62]:
int_cols   = ['product_id','brand_id','review_id','customer_id',
              'rating','thank_count','review_count','order_count',
              'discount','discount_rate']
float_cols = ['price','list_price']
dt_cols    = ['created_at','purchased_at']

# ── dtype trước
all_cast_cols = [c for c in int_cols + float_cols + dt_cols if c in df.columns]
dtype_before = df[all_cast_cols].dtypes.rename('dtype_TRƯỚC')
sample_before = df[all_cast_cols].apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None).rename('mẫu_TRƯỚC')

# ── cast
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

for col in float_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

def parse_ts(series):
    numeric  = pd.to_numeric(series, errors='coerce')
    from_ts  = pd.to_datetime(numeric, unit='s', errors='coerce', utc=True)
    from_str = pd.to_datetime(series,  errors='coerce', utc=True)
    result   = from_ts if from_ts.notna().sum() >= from_str.notna().sum() else from_str
    return result.dt.tz_localize(None)

for col in dt_cols:
    if col in df.columns:
        df[col] = parse_ts(df[col])

# ── so sánh trước / sau
dtype_after  = df[all_cast_cols].dtypes.rename('dtype_SAU')
sample_after = df[all_cast_cols].apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None).rename('mẫu_SAU')
compare = pd.concat([dtype_before, dtype_after, sample_before, sample_after], axis=1)
display(compare)


/tmp/ipykernel_2534/3448791917.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  from_str = pd.to_datetime(series,  errors='coerce', utc=True)
/tmp/ipykernel_2534/3448791917.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  from_str = pd.to_datetime(series,  errors='coerce', utc=True)


,dtype_TRƯỚC,dtype_SAU,mẫu_TRƯỚC,mẫu_SAU
product_id,object,Int64,279159356,279159356
brand_id,object,Int64,18802,18802
review_id,object,Int64,20237513,20237513
customer_id,object,Int64,28031939,28031939
rating,object,Int64,5,5
thank_count,object,Int64,0,0
review_count,object,Int64,5,5
order_count,object,Int64,None,None
discount,object,Int64,309000,309000
discount_rate,object,Int64,31,31


##  Bước 4 · Làm sạch Text

In [63]:
_EMOJI_RE    = re.compile(
    r'[\U00010000-\U0010ffff\U0001F600-\U0001F64F'
    r'\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF'
    r'\U0001F1E0-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]+',
    flags=re.UNICODE)
_MULTI_SPACE = re.compile(r'\s+')
_HTML_TAG    = re.compile(r'<[^>]+'+'>')

def clean_text(text):
    if pd.isna(text) or str(text).strip() in ('', 'nan', 'None'):
        return None
    t = str(text)
    t = _HTML_TAG.sub(' ', t)
    t = _EMOJI_RE.sub('', t)
    t = t.replace('\n',' ').replace('\r',' ').replace('\t',' ')
    t = _MULTI_SPACE.sub(' ', t)
    return t.strip() or None

text_cols = ['product_name','brand_name','title','content','customer_name']

# ── snapshot trước
snap_before = {}
for col in text_cols:
    if col in df.columns:
        sample = df[col].dropna()
        dirty  = sample[sample.str.contains(r'\s{2,}|<', regex=True, na=False)]
        snap_before[col] = (dirty.iloc[0] if not dirty.empty
                            else sample.iloc[0] if not sample.empty else '')

# ── clean
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].apply(clean_text)

# ── so sánh trước / sau
rows = []
for col in text_cols:
    if col in df.columns:
        sample    = df[col].dropna()
        after_val = sample.iloc[0] if not sample.empty else ''
        null_cnt  = df[col].isna().sum()
        rows.append({
            'cột': col,
            'TRƯỚC (mẫu)': str(snap_before.get(col,''))[:70],
            'SAU (mẫu)':   str(after_val)[:70],
            'null sau': null_cnt
        })
display(pd.DataFrame(rows).set_index('cột'))


,TRƯỚC (mẫu),SAU (mẫu),null sau
cột,,,
product_name,Tai nghe nhét tai jack cắm Type-C có mic hoco....,Đồng hồ Samsung Galaxy Fit3 - Hàng chính hãng,0
brand_name,Samsung,Samsung,0
title,Cực kì hài lòng,Cực kì hài lòng,0
content,Rất hài lòng & thật sự Cảm kích & xin chân thà...,Phù hợp với nhu cầu theo dõi sức khỏe và vận đ...,9724
customer_name,Trần Thanh Tùng,Nguyen Quoc Phong,10


## Bước 5 · Xử lý Missing Values

In [64]:
# ── missing TRƯỚC
miss_before = df.isna().sum().rename('TRƯỚC')

# ── fillna
fills = {
    'title':         '',
    'content':       '',
    'thank_count':   0,
    'customer_name': 'Ẩn danh',
}
for col, val in fills.items():
    if col in df.columns:
        df[col] = df[col].fillna(val)

# ── xóa dòng thiếu bắt buộc
required = ['review_id', 'product_id', 'rating']
b = len(df)
df = df.dropna(subset=required).reset_index(drop=True)
print(f'✂ Xóa dòng thiếu {required}: -{b - len(df):,} dòng  →  còn {len(df):,}')

# ── missing SAU
miss_after = df.isna().sum().rename('SAU')
compare = pd.concat([miss_before, miss_after], axis=1)
compare = compare[compare.sum(axis=1) > 0]
compare['thay đổi'] = compare['SAU'] - compare['TRƯỚC']
print('\nSo sánh missing trước / sau:')
display(compare if not compare.empty else pd.DataFrame({'kết quả': ['Không còn missing']}))


✂ Xóa dòng thiếu ['review_id', 'product_id', 'rating']: -0 dòng  →  còn 15,350

So sánh missing trước / sau:


,TRƯỚC,SAU,thay đổi
order_count,15350,15350,0
content,9724,0,-9724
customer_name,10,0,-10
purchased_at,16,16,0


## Bước 6 · Lọc Outlier / Bất thường

In [65]:
print(f'Trước: {len(df):,} dòng')

# ── rating
bad = df[~df['rating'].isin(VALID_RATINGS)]
if not bad.empty:
    print(f'  Rating không hợp lệ: {bad["rating"].value_counts().to_dict()}')
    display(bad[['review_id','rating','content']].head(3))
b = len(df)
df = df[df['rating'].isin(VALID_RATINGS)]
print(f'  ✂ Rating ngoài {VALID_RATINGS}: -{b-len(df):,}')

# ── content length
df['_clen'] = df['content'].str.len()
print('\nPhân phối độ dài content (ký tự):')
display(df['_clen'].describe().round(0).to_frame().T)

short = df[df['_clen'] < MIN_CONTENT_LEN][['content','rating']]
if not short.empty:
    print(f'\nMẫu content quá ngắn (< {MIN_CONTENT_LEN} ký tự):')
    display(short.head(5))

b = len(df)
df = df[df['_clen'] >= MIN_CONTENT_LEN].drop(columns=['_clen'])
print(f'  ✂ Content < {MIN_CONTENT_LEN} ký tự : -{b-len(df):,}')

print(f'\nSau : {len(df):,} dòng')
df = df.reset_index(drop=True)

Trước: 15,350 dòng
  ✂ Rating ngoài {1, 2, 3, 4, 5}: -0

Phân phối độ dài content (ký tự):


,count,mean,std,min,25%,50%,75%,max
_clen,15350.0,20.0,58.0,0.0,0.0,0.0,18.0,2189.0



Mẫu content quá ngắn (< 5 ký tự):


,content,rating
3,,5
6,,5
7,,5
10,,5
11,,5


  ✂ Content < 5 ký tự : -10,490

Sau : 4,860 dòng


## Bước 7 · Feature Engineering

In [66]:
cols_before = list(df.columns)

# NOTE: Cột 'content_len' đã bị loại bỏ theo yêu cầu của bạn.
# df['content_len'] = df['content'].str.len()

# sentiment
mapping = {1:'very_negative', 2:'negative', 3:'neutral', 4:'positive', 5:'very_positive'}
df['sentiment'] = df['rating'].map(mapping)

new_cols = [c for c in df.columns if c not in cols_before]
print(f'Cột mới: {new_cols}')
print(f'Tổng cột: {len(cols_before)} → {df.shape[1]}')

# ── preview feature mới
print('\nPhân bố sentiment:')
display(df['sentiment'].value_counts().to_frame())

Cột mới: ['sentiment']
Tổng cột: 19 → 20

Phân bố sentiment:


,count
sentiment,
very_positive,3682
positive,484
very_negative,387
neutral,189
negative,118


##  Bước 8 · Loại bỏ cột không cần thiết

In [67]:
cols_to_drop = [
    'thank_count',
    'review_count',
    'order_count'

]

# Lọc ra các cột thực sự tồn tại trong DataFrame
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]

print(f'Trước: {df.shape[1]} cột')

df = df.drop(columns=existing_cols_to_drop)

print(f'Sau: {df.shape[1]} cột')
print(f'Các cột đã loại bỏ: {existing_cols_to_drop}')

Trước: 20 cột
Sau: 17 cột
Các cột đã loại bỏ: ['thank_count', 'review_count', 'order_count']


## Bước 8 · Xuất file sạch

In [68]:
df.to_csv(OUTPUT_CLEAN, index=False, encoding='utf-8-sig')
size_kb = os.path.getsize(OUTPUT_CLEAN) // 1024
print(f'Đã lưu: {OUTPUT_CLEAN}')
print(f'   {len(df):,} dòng  |  {df.shape[1]} cột  |  ~{size_kb:,} KB')

# ── tổng kết
print(f'\n── Tổng kết ──')
print(f'Sản phẩm unique : {df["product_id"].nunique():,}')
if 'brand_name' in df.columns:
    print(f'Brand unique    : {df["brand_name"].nunique():,}')
print(f'Rating trung bình: {df["rating"].mean():.2f} / 5')

print('\nPhân bố rating:')
display(df['rating'].value_counts().sort_index().rename('count').to_frame())

if 'review_year' in df.columns:
    print('\nReviews theo năm:')
    display(df['review_year'].value_counts().sort_index().rename('count').to_frame())

print(f'\nPreview 5 dòng cuối của data sạch:')
display(df.tail())


Đã lưu: tiki_reviews_clean.csv
   4,860 dòng  |  17 cột  |  ~1,550 KB

── Tổng kết ──
Sản phẩm unique : 73
Brand unique    : 35
Rating trung bình: 4.43 / 5

Phân bố rating:


,count
rating,
1,387
2,118
3,189
4,484
5,3682



Preview 5 dòng cuối của data sạch:


,product_id,product_name,brand_id,brand_name,price,list_price,discount,discount_rate,review_id,rating,title,content,created_at,customer_id,customer_name,purchased_at,sentiment
4855,2462825,Chuột game không dây Lightspeed Logitech G304 ...,18984,Logitech,750000,1170000,420000,36,1557346,3,Bình thường,Nút cuộn giữa tiếng nhựa rất ồn :D Đây là tính...,2018-06-22 13:43:22,1524268,Hoàng Thanh Bình,2018-06-07 17:07:25,neutral
4856,2462825,Chuột game không dây Lightspeed Logitech G304 ...,18984,Logitech,750000,1170000,420000,36,1616798,5,Ngon,"Chuột chính hãng, nguyên seal, dùng rất mượt. ...",2018-07-16 14:39:20,1156386,Trần Dũng,2018-07-13 05:25:13,very_positive
4857,2462825,Chuột game không dây Lightspeed Logitech G304 ...,18984,Logitech,750000,1170000,420000,36,1593758,5,Cực kì hài lòng,Chuột chính hãng tiki giao hàng đúng hẹn. Chơi...,2018-07-08 07:25:56,1193632,Như Ý Đỗ,2018-06-11 14:29:48,very_positive
4858,2462825,Chuột game không dây Lightspeed Logitech G304 ...,18984,Logitech,750000,1170000,420000,36,1651686,4,Nhẹ vừa tay,"Cầm nhẹ, phù hợp để làm việc và gaming, pad ch...",2018-07-29 07:41:20,2129281,Đào Ngọc Anh,2018-07-20 04:47:09,positive
4859,2462825,Chuột game không dây Lightspeed Logitech G304 ...,18984,Logitech,750000,1170000,420000,36,1664894,4,Sản phẩm tốt,"Chuột nhạy, độ tùy chỉnh cao. Hỗ trợ chơi game...",2018-08-03 06:30:23,657637,Nhân HQ,2018-06-18 03:09:03,positive
